In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Adjust these if your mount path is different
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
GRAPH_PATH = DATA_DIR / "graph.nt"  # or graph.tsv
ENTITY_IDS_PATH = DATA_DIR / "embeddings/entity_ids.del"
RELATION_IDS_PATH = DATA_DIR / "embeddings/relation_ids.del"
IMAGES_PATH = DATA_DIR / "additional/images.json"

print(f"Checking data in: {DATA_DIR}\n")

# ==========================================
# 2. CHECK ENTITY ID MAPPING
# ==========================================
print("--- 2. Analyzing Entity IDs (entity_ids.del) ---")
try:
    # Read raw lines to detect header/format issues
    with open(ENTITY_IDS_PATH, "r", encoding="utf-8") as f:
        e_lines = f.readlines()
    
    print(f"Total lines: {len(e_lines)}")
    print(f"First 3 lines:\n{''.join(e_lines[:3])}")
    
    # Parse
    entity_map = {} # URI -> Index
    max_idx = -1
    for line in e_lines:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            try:
                idx = int(parts[0])
                uri = parts[1]
                entity_map[uri] = idx
                if idx > max_idx: max_idx = idx
            except ValueError:
                print(f"⚠️ Warning: Non-integer index found: {line.strip()}")
                
    print(f"Valid Entities Parsed: {len(entity_map)}")
    print(f"Max Index: {max_idx}")
    print(f"Is contiguous? {len(entity_map) == (max_idx + 1)}")
    
except Exception as e:
    print(f"❌ Error reading entity_ids: {e}")

# ==========================================
# 3. CHECK RELATION ID MAPPING
# ==========================================
print("\n--- 3. Analyzing Relation IDs (relation_ids.del) ---")
try:
    with open(RELATION_IDS_PATH, "r", encoding="utf-8") as f:
        r_lines = f.readlines()
        
    print(f"Total lines: {len(r_lines)}")
    print(f"First 3 lines:\n{''.join(r_lines[:3])}")
    
    relation_map = {} # URI -> Index
    for line in r_lines:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            try:
                idx = int(parts[0])
                uri = parts[1]
                relation_map[uri] = idx
            except: pass
            
    print(f"Valid Relations Parsed: {len(relation_map)}")
    
except Exception as e:
    print(f"❌ Error reading relation_ids: {e}")

# ==========================================
# 4. CHECK GRAPH STRUCTURE (graph.nt)
# ==========================================
print(f"\n--- 4. Analyzing Knowledge Graph ({GRAPH_PATH.name}) ---")
try:
    # We use manual parsing for speed/control instead of rdflib for this audit
    triple_count = 0
    unknown_entities = set()
    unknown_relations = set()
    relation_counts = Counter()
    
    # Sample vars
    sample_triples = []
    
    with open(GRAPH_PATH, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            
            # Simple N-Triple parser (split by space, handle <>)
            # This is a heuristic parser; standard NT is '<s> <p> <o> .'
            parts = line.split(" ")
            if len(parts) < 3: continue
            
            s = parts[0].strip("<>")
            p = parts[1].strip("<>")
            o = parts[2].strip("<>") # Object might be literal "string"@en
            
            # Record stats
            triple_count += 1
            relation_counts[p] += 1
            
            # Check coverage (Only check S and P, and O if it's a URI)
            if s not in entity_map: unknown_entities.add(s)
            if p not in relation_map: unknown_relations.add(p)
            
            # Heuristic: if Object starts with http, treat as entity
            if o.startswith("http") and o not in entity_map:
                unknown_entities.add(o)
                
            if i < 3: sample_triples.append(f"{s} --[{p}]--> {o}")

    print(f"Total Triples Scanned: {triple_count}")
    print(f"Sample Triples:\n" + "\n".join(sample_triples))
    
    print(f"\n[Coverage Report]")
    print(f"Unknown Entities found in Graph: {len(unknown_entities)}")
    if len(unknown_entities) > 0:
        print(f"Example Unknown Entities: {list(unknown_entities)[:5]}")
        
    print(f"Unknown Relations found in Graph: {len(unknown_relations)}")
    if len(unknown_relations) > 0:
        print(f"Example Unknown Relations: {list(unknown_relations)[:5]}")

    print(f"\n[Top 10 Relations by Frequency]")
    for r, c in relation_counts.most_common(10):
        print(f"  {r}: {c}")

except Exception as e:
    print(f"❌ Error reading graph: {e}")

# ==========================================
# 5. CHECK MULTIMEDIA (images.json)
# ==========================================
print("\n--- 5. Analyzing Multimedia (images.json) ---")
try:
    if IMAGES_PATH.exists():
        with open(IMAGES_PATH, "r") as f:
            imgs = json.load(f)
        print(f"Total Image Entries: {len(imgs)}")
        if len(imgs) > 0:
            print(f"Sample Entry: {imgs[0]}")
            
        # Check alignment: Do the image IDs (movies/cast) exist in our Entity Map?
        matched = 0
        unmatched = 0
        for item in imgs[:1000]: # Check first 1000 for speed
            # Items usually have 'movie' or 'cast' keys with URIs
            keys_to_check = ['movie', 'cast', 'id']
            found_uri = False
            for k in keys_to_check:
                if k in item:
                    uris = item[k] if isinstance(item[k], list) else [item[k]]
                    for u in uris:
                        # Strip <> just in case
                        u_clean = u.replace("<", "").replace(">", "")
                        if u_clean in entity_map:
                            found_uri = True
            
            if found_uri: matched += 1
            else: unmatched += 1
            
        print(f"Alignment Check (Sample 1000): {matched} matched / {unmatched} unmatched")
    else:
        print("❌ images.json not found.")
except Exception as e:
    print(f"❌ Error reading images.json: {e}")

Checking data in: /space_mounts/atai-hs25/dataset

--- 2. Analyzing Entity IDs (entity_ids.del) ---
Total lines: 175660
First 3 lines:
0	http://www.wikidata.org/entity/Q0058036710383292
1	http://www.wikidata.org/entity/Q0063555679015095
2	http://www.wikidata.org/entity/Q0065575877878203

Valid Entities Parsed: 175660
Max Index: 175659
Is contiguous? True

--- 3. Analyzing Relation IDs (relation_ids.del) ---
Total lines: 491
First 3 lines:
0	http://ddis.ch/atai/indirectSubclassOf
1	http://www.wikidata.org/prop/direct/P101
2	http://www.wikidata.org/prop/direct/P1018

Valid Relations Parsed: 491

--- 4. Analyzing Knowledge Graph (graph.nt) ---
Total Triples Scanned: 2413825
Sample Triples:
http://www.wikidata.org/entity/Q1390537 --[http://www.wikidata.org/prop/direct/P1412]--> http://www.wikidata.org/entity/Q652
http://www.wikidata.org/entity/Q2564319 --[http://www.wikidata.org/prop/direct/P364]--> http://www.wikidata.org/entity/Q1860
http://www.wikidata.org/entity/Q179059 --[http://www.w

# Retraining the Embeddings

## Setup & Data Loading

In [2]:
import os
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict

# --- CONFIGURATION ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
OUTPUT_DIR = Path("embeddings")  # Will save to local folder
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

ENTITY_ID_PATH = DATA_DIR / "embeddings/entity_ids.del"
RELATION_ID_PATH = DATA_DIR / "embeddings/relation_ids.del"
GRAPH_PATH = DATA_DIR / "graph.nt"

# --- 1. LOAD ID MAPPINGS ---
print("Loading ID Mappings...")
entity_to_idx = {}
with open(ENTITY_ID_PATH, "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            entity_to_idx[parts[1]] = int(parts[0])

relation_to_idx = {}
with open(RELATION_ID_PATH, "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            relation_to_idx[parts[1]] = int(parts[0])

num_entities = len(entity_to_idx)
num_relations = len(relation_to_idx)
print(f"Entities: {num_entities}, Relations: {num_relations}")

# --- 2. LOAD TRIPLES ---
print("Loading Graph and converting to integers...")
triples = []
skipped = 0

with open(GRAPH_PATH, "r") as f:
    for line in f:
        parts = line.strip().split(" ")
        if len(parts) < 3: continue
        
        # Strip brackets <uri> -> uri
        s_uri = parts[0].strip("<>")
        p_uri = parts[1].strip("<>")
        o_uri = parts[2].strip("<>")
        
        # Check if all exist in our mappings
        if (s_uri in entity_to_idx) and (p_uri in relation_to_idx) and (o_uri in entity_to_idx):
            s_idx = entity_to_idx[s_uri]
            r_idx = relation_to_idx[p_uri]
            o_idx = entity_to_idx[o_uri]
            triples.append([s_idx, r_idx, o_idx])
        else:
            skipped += 1

triples_tensor = torch.tensor(triples, dtype=torch.long)
print(f"Loaded {len(triples)} training triples.")
print(f"Skipped {skipped} triples (schema/metadata/literals).")

Loading ID Mappings...
Entities: 175660, Relations: 491
Loading Graph and converting to integers...
Loaded 2009173 training triples.
Skipped 404652 triples (schema/metadata/literals).


## Model Definition (TransE)

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, dim=128, margin=1.0):
        super(TransE, self).__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.dim = dim
        self.margin = margin
        
        # Embeddings
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        
        # Initialization (Xavier)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
    def forward(self, h, r, t):
        # Normalize every step to ensure cosine similarity works later
        h_vec = F.normalize(self.entity_emb(h), p=2, dim=1)
        r_vec = F.normalize(self.relation_emb(r), p=2, dim=1)
        t_vec = F.normalize(self.entity_emb(t), p=2, dim=1)
        
        # Distance: ||h + r - t||
        # We use L1 or L2. L2 is often smoother.
        score = torch.norm(h_vec + r_vec - t_vec, p=2, dim=1)
        return score
    
    def get_entity_embeddings(self):
        return F.normalize(self.entity_emb.weight.data, p=2, dim=1).cpu().numpy()

    def get_relation_embeddings(self):
        return F.normalize(self.relation_emb.weight.data, p=2, dim=1).cpu().numpy()

## Training Loop

In [4]:
from torch.utils.data import DataLoader, TensorDataset
import time

# --- HYPERPARAMETERS ---
DIM = 128          # 128 is sufficient for this data size and keeps file size low
BATCH_SIZE = 4096
LR = 0.01
EPOCHS = 15        # Should converge quickly given the graph density
MARGIN = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {device}")

# Prepare Data
train_dataset = TensorDataset(triples_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize Model
model = TransE(num_entities, num_relations, dim=DIM, margin=MARGIN)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Training Loop
start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    model.train()
    
    for batch in train_loader:
        # Positive samples
        h, r, t = batch[0][:, 0].to(device), batch[0][:, 1].to(device), batch[0][:, 2].to(device)
        
        # Negative sampling (Corrupt head or tail)
        # 50% chance to corrupt head, 50% tail
        corrupt_head_prob = torch.rand(h.size(0), device=device)
        random_entities = torch.randint(0, num_entities, (h.size(0),), device=device)
        
        h_neg = torch.where(corrupt_head_prob > 0.5, random_entities, h)
        t_neg = torch.where(corrupt_head_prob > 0.5, t, random_entities)
        
        optimizer.zero_grad()
        
        pos_score = model(h, r, t)
        neg_score = model(h_neg, r, t_neg)
        
        # Margin Ranking Loss: max(0, margin + pos - neg)
        # We want pos_score (distance) to be small, neg_score to be large
        loss = torch.mean(torch.relu(MARGIN + pos_score - neg_score))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {total_loss:.4f} | Time: {time.time() - start_time:.1f}s")

# --- SAVE ---
print("\nSaving Embeddings...")
np.save(OUTPUT_DIR / "RFC_entity_embeds.npy", model.get_entity_embeddings())
np.save(OUTPUT_DIR / "RFC_relation_embeds.npy", model.get_relation_embeddings())
print(f"✅ Saved RFC_entity_embeds.npy and RFC_relation_embeds.npy to {OUTPUT_DIR}")

Training on cuda
Epoch 1/15 | Loss: 327.6233 | Time: 15.0s
Epoch 2/15 | Loss: 218.4262 | Time: 27.8s
Epoch 3/15 | Loss: 173.6095 | Time: 41.3s
Epoch 4/15 | Loss: 154.0520 | Time: 54.1s
Epoch 5/15 | Loss: 143.1164 | Time: 67.6s
Epoch 6/15 | Loss: 136.9213 | Time: 80.4s
Epoch 7/15 | Loss: 133.1187 | Time: 93.9s
Epoch 8/15 | Loss: 130.7370 | Time: 107.4s
Epoch 9/15 | Loss: 128.6423 | Time: 120.2s
Epoch 10/15 | Loss: 127.0619 | Time: 133.7s
Epoch 11/15 | Loss: 126.0320 | Time: 146.5s
Epoch 12/15 | Loss: 125.0381 | Time: 160.0s
Epoch 13/15 | Loss: 124.0449 | Time: 172.8s
Epoch 14/15 | Loss: 123.1254 | Time: 186.3s
Epoch 15/15 | Loss: 122.5758 | Time: 199.1s

Saving Embeddings...
✅ Saved RFC_entity_embeds.npy and RFC_relation_embeds.npy to embeddings


# Data Check

In [5]:
import json
import os
from pathlib import Path
from rdflib import Graph, URIRef

# --- CONFIGURATION ---
# Ensure this matches your actual mount path
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
GRAPH_PATH = DATA_DIR / "graph.nt"
IMAGES_PATH = DATA_DIR / "additional/images.json"

# --- TEST DATA FROM FINAL EVALUATION.TXT ---

# 1. One-hop Questions (Entity, Property ID, Description)
qa_targets = [
    ("Aro Tolbukhin. En la mente del asesino", "P495", "Country"),
    ("Shortcut to Happiness", "P58", "Screenwriter"),
    ("Fargo", "P57", "Director"),
    ("Bandit Queen", "P136", "Genre"),
    ("Miracles Still Happen", "P577", "Publication Date"),
    ("Apocalypse Now", "P57", "Director"),
    ("12 Monkeys", "P58", "Screenwriter"),
    ("Shoplifters", "P136", "Genre"),
    ("Good Will Hunting", "P57", "Director"),
    ("Garden State", "P57", "Director"),
    ("Enchanted April", "P86", "Composer"),
    ("Midnight in Paris", "P1411", "Nominated For"),
    ("Forrest Gump", "P57", "Director"),
    ("The Longest Day", "P57", "Director"),
    ("Captain America: Civil War", "P57", "Director"),
    ("The Ascent", "P57", "Director")
]

# 2. Recommendation Entities (Just check if they exist as Movies)
rec_targets = [
    "Singin' in the Rain", "Moulin Rouge",
    "La Dolce Vita", "The Voice of the Moon",
    "Chicago", "Memoirs of a Geisha", "Alice in Wonderland",
    "The Lord of the Rings: The Fellowship of the Ring",
    "Twin Sisters of Kyoto",
    "Meryl Streep" # Person check
]

# 3. Multimedia Targets (Entity, Expected ID fragment)
mm_targets = [
    ("Halle Berry", "knLBjlWP36OSu1A0Btbyy9r1WXH"),
    ("Denzel Washington", "ffI7oAkIBvXpAMxNn21VDAnna1g"),
    ("Forest Gump", "4bZr216gYuanA1urxFC561FmsIp"), # Typo in doc?
    ("Forrest Gump", "4bZr216gYuanA1urxFC561FmsIp")  # Correct spelling
]

# --- LOAD GRAPH ---
print(f"Loading Graph from {GRAPH_PATH} ... (This may take 30-60s)")
g = Graph()
try:
    g.parse(GRAPH_PATH, format="nt")
    print(f"✅ Graph loaded. Total triples: {len(g)}")
except Exception as e:
    print(f"❌ Failed to load graph: {e}")
    exit()

# --- CHECK 1: ONE-HOP QA ---
print("\n" + "="*60)
print("PART 1: ONE-HOP QA DATA CHECK")
print("="*60)

for name, prop, prop_name in qa_targets:
    print(f"\n🔍 Checking: '{name}' -> {prop_name} ({prop})")
    
    # 1. Find Entity URI by Label
    # Using simple string matching in SPARQL for flexibility
    query_label = f"""
        SELECT ?s WHERE {{
            ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label .
            FILTER(LCASE(STR(?label)) = LCASE("{name}"))
        }}
    """
    results = list(g.query(query_label))
    
    if not results:
        print(f"   ❌ Entity NOT FOUND (No label match)")
        continue
    
    uri = str(results[0][0])
    print(f"   ✅ Found URI: {uri}")
    
    # 2. Check Property
    query_prop = f"""
        SELECT ?o ?label WHERE {{
            <{uri}> <http://www.wikidata.org/prop/direct/{prop}> ?o .
            OPTIONAL {{ ?o <http://www.w3.org/2000/01/rdf-schema#label> ?label . FILTER(LANG(?label) = 'en') }}
        }}
    """
    res_prop = list(g.query(query_prop))
    
    if res_prop:
        print(f"   ✅ Property FOUND ({len(res_prop)} values):")
        for row in res_prop:
            val = str(row[0]).split('/')[-1]
            lbl = str(row[1]) if row[1] else "No Label"
            print(f"      - ID: {val} | Label: {lbl}")
    else:
        print(f"   ❌ Property MISSING (Edge wdt:{prop} does not exist)")

# --- CHECK 2: RECOMMENDATION ENTITIES ---
print("\n" + "="*60)
print("PART 2: RECOMMENDATION ENTITY CHECK")
print("="*60)

for name in rec_targets:
    query = f"""
        SELECT ?s ?type WHERE {{
            ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label .
            FILTER(LCASE(STR(?label)) = LCASE("{name}"))
            OPTIONAL {{ ?s <http://www.wikidata.org/prop/direct/P31> ?type . }}
        }}
    """
    res = list(g.query(query))
    if res:
        uri = str(res[0][0])
        type_uri = str(res[0][1]) if res[0][1] else "Unknown"
        print(f"✅ Found '{name}': {uri} (Type: {type_uri.split('/')[-1]})")
    else:
        print(f"❌ '{name}' NOT FOUND in Graph")

# --- CHECK 3: MULTIMEDIA ---
print("\n" + "="*60)
print("PART 3: MULTIMEDIA INDEX CHECK")
print("="*60)

if IMAGES_PATH.exists():
    with open(IMAGES_PATH, "r") as f:
        img_data = json.load(f)
    
    for name, expected_id in mm_targets:
        print(f"\n🔍 Searching image for '{name}'...")
        
        # We need the URI first to check strictly, but let's fuzzy search the JSON values
        # since we might not have the URI if the graph check failed above.
        found_in_json = False
        
        # 1. Try to get URI from graph to cross-reference
        query_label = f"""SELECT ?s WHERE {{ ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label . FILTER(LCASE(STR(?label)) = LCASE("{name}")) }}"""
        res = list(g.query(query_label))
        uri_from_graph = str(res[0][0]) if res else None
        
        for item in img_data:
            # Check image ID directly (reverse check)
            if expected_id in item.get("img", ""):
                print(f"   ✅ Found entry matching expected ID '{expected_id}':")
                print(f"      - Img: {item.get('img')}")
                print(f"      - Linked URIs: {item.get('movie') or item.get('cast') or item.get('id')}")
                found_in_json = True
                break
                
            # Check by URI if we have one
            if uri_from_graph:
                vals = []
                for k in ['movie', 'cast', 'id']:
                    v = item.get(k)
                    if isinstance(v, list): vals.extend(v)
                    elif isinstance(v, str): vals.append(v)
                
                clean_graph_uri = uri_from_graph.replace("<","").replace(">","")
                if clean_graph_uri in vals:
                    print(f"   ✅ Found entry matching Graph URI {uri_from_graph}:")
                    print(f"      - Img: {item.get('img')}")
                    found_in_json = True
                    break
        
        if not found_in_json:
            print(f"   ❌ NO IMAGE found for '{name}' (Expected ID: {expected_id})")

else:
    print("❌ images.json not found")

Loading Graph from /space_mounts/atai-hs25/dataset/graph.nt ... (This may take 30-60s)
✅ Graph loaded. Total triples: 2413825

PART 1: ONE-HOP QA DATA CHECK

🔍 Checking: 'Aro Tolbukhin. En la mente del asesino' -> Country (P495)
   ✅ Found URI: http://www.wikidata.org/entity/Q4795458
   ✅ Property FOUND (1 values):
      - ID: Q96 | Label: No Label

🔍 Checking: 'Shortcut to Happiness' -> Screenwriter (P58)
   ✅ Found URI: http://www.wikidata.org/entity/Q1115508
   ✅ Property FOUND (2 values):
      - ID: Q458445 | Label: No Label
      - ID: Q361336 | Label: No Label

🔍 Checking: 'Fargo' -> Director (P57)
   ✅ Found URI: http://www.wikidata.org/entity/Q222720
   ✅ Property FOUND (2 values):
      - ID: Q13595531 | Label: No Label
      - ID: Q13595311 | Label: No Label

🔍 Checking: 'Bandit Queen' -> Genre (P136)
   ✅ Found URI: http://www.wikidata.org/entity/Q2762808
   ✅ Property FOUND (3 values):
      - ID: Q130232 | Label: No Label
      - ID: Q645928 | Label: No Label
      - ID: 

# generate 3 essential artifacts:

RFC_entity_embeds.npy (The brain)

entity_labels.json (The dictionary - fixes "No Label")

imdb_map.json (The bridge - fixes Multimedia "No Image")

In [6]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from rdflib import Graph
from torch.utils.data import DataLoader, TensorDataset
import time

# --- CONFIGURATION ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
OUTPUT_DIR = Path("embeddings") 
METADATA_DIR = Path("metadata")

OUTPUT_DIR.mkdir(exist_ok=True)
METADATA_DIR.mkdir(exist_ok=True)

GRAPH_PATH = DATA_DIR / "graph.nt"
ENTITY_ID_PATH = DATA_DIR / "embeddings/entity_ids.del"
RELATION_ID_PATH = DATA_DIR / "embeddings/relation_ids.del"

# ==========================================
# PART 1: DATA EXTRACTION (Labels & IDs)
# ==========================================
print("\n=== PART 1: Extracting Metadata from Graph ===")
print(f"Loading {GRAPH_PATH}...")

# We use raw parsing for speed (rdflib can be slow for full scans)
entity_labels = {}
imdb_map = {}

# Standard URIs
P_LABEL = "http://www.w3.org/2000/01/rdf-schema#label"
P_IMDB = "http://www.wikidata.org/prop/direct/P345"

with open(GRAPH_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(" ")
        if len(parts) < 3: continue
        
        s = parts[0].strip("<>")
        p = parts[1].strip("<>")
        
        # Extract Labels
        if p == P_LABEL:
            # Object is literal: "Label"@en
            # Find the string content
            o_raw = " ".join(parts[2:-1]) # Rejoin if spaces in name
            if "@en" in o_raw:
                label = o_raw.split('"')[1]
                entity_labels[s] = label
            elif s not in entity_labels: # Fallback to non-english if no english yet
                try:
                    label = o_raw.split('"')[1]
                    entity_labels[s] = label
                except: pass

        # Extract IMDb IDs (Bridge for Images)
        if p == P_IMDB:
            o = parts[2].strip('<>').replace('"', '')
            imdb_map[s] = o

print(f"✅ Extracted {len(entity_labels)} labels.")
print(f"✅ Extracted {len(imdb_map)} IMDb mappings.")

# Save to JSON
with open(METADATA_DIR / "entity_labels.json", "w") as f:
    json.dump(entity_labels, f)
with open(METADATA_DIR / "imdb_map.json", "w") as f:
    json.dump(imdb_map, f)
    
print(f"Saved metadata to {METADATA_DIR}")

# ==========================================
# PART 2: EMBEDDING TRAINING
# ==========================================
print("\n=== PART 2: Training Embeddings ===")

# 1. Load IDs
entity_to_idx = {}
with open(ENTITY_ID_PATH, "r") as f:
    for line in f:
        p = line.strip().split("\t")
        if len(p) == 2: entity_to_idx[p[1]] = int(p[0])

relation_to_idx = {}
with open(RELATION_ID_PATH, "r") as f:
    for line in f:
        p = line.strip().split("\t")
        if len(p) == 2: relation_to_idx[p[1]] = int(p[0])

num_ent = len(entity_to_idx)
num_rel = len(relation_to_idx)

# 2. Load Triples
triples = []
with open(GRAPH_PATH, "r") as f:
    for line in f:
        p = line.strip().split(" ")
        if len(p) < 3: continue
        s, pr, o = p[0].strip("<>"), p[1].strip("<>"), p[2].strip("<>")
        if s in entity_to_idx and pr in relation_to_idx and o in entity_to_idx:
            triples.append([entity_to_idx[s], relation_to_idx[pr], entity_to_idx[o]])

triples_tensor = torch.tensor(triples, dtype=torch.long)
print(f"Training on {len(triples)} triples.")

# 3. Model
class TransE(nn.Module):
    def __init__(self, num_e, num_r, dim=128):
        super().__init__()
        self.e_emb = nn.Embedding(num_e, dim)
        self.r_emb = nn.Embedding(num_r, dim)
        nn.init.xavier_uniform_(self.e_emb.weight)
        nn.init.xavier_uniform_(self.r_emb.weight)
    
    def forward(self, h, r, t):
        h = F.normalize(self.e_emb(h), p=2, dim=1)
        r = F.normalize(self.r_emb(r), p=2, dim=1)
        t = F.normalize(self.e_emb(t), p=2, dim=1)
        return torch.norm(h + r - t, p=2, dim=1)

# 4. Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransE(num_ent, num_rel).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loader = DataLoader(TensorDataset(triples_tensor), batch_size=8192, shuffle=True)

print("Starting Training (15 Epochs)...")
for epoch in range(15):
    total_loss = 0
    model.train()
    for batch in loader:
        h, r, t = batch[0][:,0].to(device), batch[0][:,1].to(device), batch[0][:,2].to(device)
        
        # Negative Sampling
        rnd = torch.randint(0, num_ent, h.size(), device=device)
        mask = torch.rand(h.size(), device=device) > 0.5
        h_n = torch.where(mask, rnd, h)
        t_n = torch.where(mask, t, rnd)
        
        pos = model(h, r, t)
        neg = model(h_n, r, t_n)
        
        loss = torch.mean(torch.relu(1.0 + pos - neg))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# 5. Save
ent_emb = F.normalize(model.e_emb.weight.data.cpu(), p=2, dim=1).numpy()
rel_emb = F.normalize(model.r_emb.weight.data.cpu(), p=2, dim=1).numpy()

np.save(OUTPUT_DIR / "RFC_entity_embeds.npy", ent_emb)
np.save(OUTPUT_DIR / "RFC_relation_embeds.npy", rel_emb)
print("✅ Embeddings Saved.")


=== PART 1: Extracting Metadata from Graph ===
Loading /space_mounts/atai-hs25/dataset/graph.nt...
✅ Extracted 129673 labels.
✅ Extracted 95569 IMDb mappings.
Saved metadata to metadata

=== PART 2: Training Embeddings ===
Training on 2009173 triples.
Starting Training (15 Epochs)...
Epoch 1 Loss: 169.0100
Epoch 2 Loss: 115.0561
Epoch 3 Loss: 88.5480
Epoch 4 Loss: 77.3255
Epoch 5 Loss: 71.5591
Epoch 6 Loss: 68.1191
Epoch 7 Loss: 66.0186
Epoch 8 Loss: 64.6114
Epoch 9 Loss: 63.5885
Epoch 10 Loss: 62.7195
Epoch 11 Loss: 62.1470
Epoch 12 Loss: 61.6399
Epoch 13 Loss: 61.2344
Epoch 14 Loss: 60.7818
Epoch 15 Loss: 60.6469
✅ Embeddings Saved.


In [7]:
import json
import os
from pathlib import Path

# --- CONFIGURATION ---
METADATA_DIR = Path("metadata")
LABELS_PATH = METADATA_DIR / "entity_labels.json"
IMDB_PATH = METADATA_DIR / "imdb_map.json"

# --- Test Cases (Q-IDs from your previous logs and Wikidata) ---
# We want to check if these URIs have Labels and IMDb IDs
test_uris = {
    "http://www.wikidata.org/entity/Q134773": "Forrest Gump",
    "http://www.wikidata.org/entity/Q1033016": "Halle Berry",
    "http://www.wikidata.org/entity/Q4090": "Denzel Washington",
    "http://www.wikidata.org/entity/Q4795458": "Aro Tolbukhin",
    "http://www.wikidata.org/entity/Q222720": "Fargo",
    "http://www.wikidata.org/entity/Q3410561": "The Longest Day",
    "http://www.wikidata.org/entity/Q760053": "The Ascent"
}

print(f"Checking metadata files in {METADATA_DIR}...\n")

# --- 1. Check Labels ---
if LABELS_PATH.exists():
    print(f"Loading {LABELS_PATH}...")
    with open(LABELS_PATH, "r") as f:
        labels = json.load(f)
    print(f"✅ Loaded {len(labels)} entity labels.")
    
    print("\n--- Verifying Specific Labels ---")
    for uri, expected_name in test_uris.items():
        # Check full URI
        if uri in labels:
            print(f"✅ Found Label for {expected_name}: '{labels[uri]}'")
        else:
            # Check just QID (sometimes stored without full URI)
            qid = uri.split("/")[-1]
            if qid in labels:
                print(f"✅ Found Label for {expected_name} (short key): '{labels[qid]}'")
            else:
                print(f"❌ MISSING Label for {expected_name} ({uri})")
else:
    print(f"❌ ERROR: {LABELS_PATH} does not exist. Extraction failed.")

# --- 2. Check IMDb Mapping (For Images) ---
if IMDB_PATH.exists():
    print(f"\nLoading {IMDB_PATH}...")
    with open(IMDB_PATH, "r") as f:
        imdb_map = json.load(f)
    print(f"✅ Loaded {len(imdb_map)} IMDb mappings.")
    
    print("\n--- Verifying IMDb IDs (Bridge to Images) ---")
    for uri, name in test_uris.items():
        if uri in imdb_map:
            print(f"✅ Found IMDb ID for {name}: '{imdb_map[uri]}'")
        else:
            # Check short key
            qid = uri.split("/")[-1]
            if qid in imdb_map:
                print(f"✅ Found IMDb ID for {name} (short key): '{imdb_map[qid]}'")
            else:
                print(f"❌ MISSING IMDb ID for {name} ({uri})")
else:
    print(f"❌ ERROR: {IMDB_PATH} does not exist. Extraction failed.")

Checking metadata files in metadata...

Loading metadata/entity_labels.json...
✅ Loaded 129673 entity labels.

--- Verifying Specific Labels ---
✅ Found Label for Forrest Gump: 'Forrest Gump'
✅ Found Label for Halle Berry: 'Halle Berry'
❌ MISSING Label for Denzel Washington (http://www.wikidata.org/entity/Q4090)
✅ Found Label for Aro Tolbukhin: 'Aro Tolbukhin. En la mente del asesino'
✅ Found Label for Fargo: 'Fargo'
✅ Found Label for The Longest Day: 'The Longest Day'
✅ Found Label for The Ascent: 'The Ascent'

Loading metadata/imdb_map.json...
✅ Loaded 95569 IMDb mappings.

--- Verifying IMDb IDs (Bridge to Images) ---
✅ Found IMDb ID for Forrest Gump: 'tt0109830^^<http://www.w3.org/2001/XMLSchema#string'
✅ Found IMDb ID for Halle Berry: 'nm0000932^^<http://www.w3.org/2001/XMLSchema#string'
❌ MISSING IMDb ID for Denzel Washington (http://www.wikidata.org/entity/Q4090)
✅ Found IMDb ID for Aro Tolbukhin: 'tt0291022^^<http://www.w3.org/2001/XMLSchema#string'
✅ Found IMDb ID for Fargo: '

# Refine Data Extraction

In [8]:
import json
from pathlib import Path

# --- CONFIGURATION ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
GRAPH_PATH = DATA_DIR / "graph.nt"
METADATA_DIR = Path("metadata")
METADATA_DIR.mkdir(exist_ok=True)

# Standard URIs
P_LABEL = "http://www.w3.org/2000/01/rdf-schema#label"
P_IMDB = "http://www.wikidata.org/prop/direct/P345"

# Targets to debug specifically
debug_uris = {
    "http://www.wikidata.org/entity/Q4090": "Denzel Washington",
    "http://www.wikidata.org/entity/Q3410561": "The Longest Day"
}

print(f"Scanning {GRAPH_PATH} with IMPROVED extraction logic...\n")

entity_labels = {}
imdb_map = {}
debug_triples = []

with open(GRAPH_PATH, "r", encoding="utf-8") as f:
    for line in f:
        # Debug: Capture raw lines for our problem entities
        for uri in debug_uris:
            if uri in line:
                debug_triples.append(line.strip())

        parts = line.strip().split(" ")
        if len(parts) < 3: continue
        
        s = parts[0].strip("<>")
        p = parts[1].strip("<>")
        
        # --- 1. Robust Label Extraction ---
        if p == P_LABEL:
            # Join parts in case name has spaces: "Denzel Washington"@en
            # parts[2:] covers the name and language tag
            obj_raw = " ".join(parts[2:-1]) 
            
            # Extract content inside quotes
            if '"' in obj_raw:
                label_content = obj_raw.split('"')[1]
                
                # Logic: Prefer @en, but accept ANY label if we don't have one yet
                if "@en" in obj_raw:
                    entity_labels[s] = label_content
                elif s not in entity_labels:
                    entity_labels[s] = label_content

        # --- 2. Robust IMDb ID Extraction (FIXED) ---
        if p == P_IMDB:
            # Raw object might be: "tt0109830"^^<http://www.w3.org/2001/XMLSchema#string>
            o_raw = parts[2]
            
            # Extract content inside quotes
            if '"' in o_raw:
                imdb_id = o_raw.split('"')[1]
                imdb_map[s] = imdb_id

print(f"✅ Extracted {len(entity_labels)} labels.")
print(f"✅ Extracted {len(imdb_map)} IMDb mappings.")

# --- Save to JSON ---
with open(METADATA_DIR / "entity_labels.json", "w") as f:
    json.dump(entity_labels, f)
with open(METADATA_DIR / "imdb_map.json", "w") as f:
    json.dump(imdb_map, f)
print(f"Saved metadata files to {METADATA_DIR}")

# --- Debug Analysis ---
print("\n=== DIAGNOSIS: Problem Entities ===")
for uri, name in debug_uris.items():
    print(f"\nChecking raw data for {name} ({uri}):")
    
    # Check what we captured in memory
    lbl = entity_labels.get(uri, "MISSING")
    imdb = imdb_map.get(uri, "MISSING")
    print(f"  -> Extracted Label: {lbl}")
    print(f"  -> Extracted IMDb:  {imdb}")
    
    # Check raw triples to see if it was our code or the data
    relevant_lines = [l for l in debug_triples if uri in l]
    if not relevant_lines:
        print("  -> ❌ NO TRIPLES FOUND in graph. This entity is completely missing.")
    else:
        print(f"  -> Found {len(relevant_lines)} raw triples. Samples:")
        for l in relevant_lines[:5]:
            print(f"     {l}")

Scanning /space_mounts/atai-hs25/dataset/graph.nt with IMPROVED extraction logic...

✅ Extracted 129673 labels.
✅ Extracted 95569 IMDb mappings.
Saved metadata files to metadata

=== DIAGNOSIS: Problem Entities ===

Checking raw data for Denzel Washington (http://www.wikidata.org/entity/Q4090):
  -> Extracted Label: MISSING
  -> Extracted IMDb:  MISSING
  -> Found 279 raw triples. Samples:
     <http://www.wikidata.org/entity/Q409064> <http://www.wikidata.org/prop/direct/P136> <http://www.wikidata.org/entity/Q188473> .
     <http://www.wikidata.org/entity/Q40909> <http://www.wikidata.org/prop/direct/P19> <http://www.wikidata.org/entity/Q84> .
     <http://www.wikidata.org/entity/Q409064> <http://www.wikidata.org/prop/direct/P179> <http://www.wikidata.org/entity/Q21646057> .
     <http://www.wikidata.org/entity/Q409064> <http://www.wikidata.org/prop/direct/P161> <http://www.wikidata.org/entity/Q106529> .
     <http://www.wikidata.org/entity/Q1050528> <http://www.wikidata.org/prop/direct

# Embedding Inference Diagnostic Script

In [1]:
import numpy as np
import faiss
from pathlib import Path

# --- CONFIG ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
CODE_DIR = Path("/files/Final Project Submission")

ENTITY_MAP_PATH = DATA_DIR / "embeddings/entity_ids.del"
RELATION_MAP_PATH = DATA_DIR / "embeddings/relation_ids.del"
ENT_EMB_PATH = CODE_DIR / "embeddings/RFC_entity_embeds.npy"
REL_EMB_PATH = CODE_DIR / "embeddings/RFC_relation_embeds.npy"

print("=== DIAGNOSIS: Embedding Inference ===\n")

# 1. Load Mappings
print("Loading Maps...")
ent_map = {}
with open(ENTITY_MAP_PATH) as f:
    for line in f:
        i, u = line.strip().split('\t')
        ent_map[u] = int(i)

rel_map = {}
with open(RELATION_MAP_PATH) as f:
    for line in f:
        i, u = line.strip().split('\t')
        rel_map[u] = int(i)

idx2ent = {v: k for k, v in ent_map.items()}

# 2. Load Embeddings
print("Loading Embeddings...")
if not ENT_EMB_PATH.exists() or not REL_EMB_PATH.exists():
    print("❌ Embeddings not found! Did you run the training cell?")
    exit()

E = np.load(ENT_EMB_PATH)
R = np.load(REL_EMB_PATH)
print(f"Entities: {E.shape}, Relations: {R.shape}")

# 3. Test Inference: "Who is the director(P57) of The Longest Day(Q3410561)?"
head_uri = "http://www.wikidata.org/entity/Q3410561" # The Longest Day
rel_uri = "http://www.wikidata.org/prop/direct/P57" # Director

print(f"\nTesting Inference for:\nHead: {head_uri}\nRel : {rel_uri}")

if head_uri not in ent_map:
    print("❌ Head Entity NOT in ID map.")
elif rel_uri not in rel_map:
    print(f"❌ Relation '{rel_uri}' NOT in ID map.")
    print("Available relations sample:", list(rel_map.keys())[:5])
else:
    print("✅ Head and Relation found in maps.")
    
    h_idx = ent_map[head_uri]
    r_idx = rel_map[rel_uri]
    
    h_vec = E[h_idx]
    r_vec = R[r_idx]
    
    # TransE: t approx h + r
    target = h_vec + r_vec
    
    # Search
    index = faiss.IndexFlatL2(E.shape[1])
    index.add(E)
    
    D, I = index.search(target.reshape(1, -1), 5)
    
    print("\nPredicted Tails (Directors?):")
    for i in range(5):
        idx = I[0][i]
        dist = D[0][i]
        uri = idx2ent.get(idx, "Unknown")
        print(f"  {i+1}. [{dist:.4f}] {uri}")
        
    print("\n💡 Logic Check: If you see entities above, the 'I think...' logic SHOULD work.")
    print("If the bot didn't output this, the bug is in 'nlq.py' (maybe predicate matching failed).")

=== DIAGNOSIS: Embedding Inference ===

Loading Maps...
Loading Embeddings...
Entities: (175660, 128), Relations: (491, 128)

Testing Inference for:
Head: http://www.wikidata.org/entity/Q3410561
Rel : http://www.wikidata.org/prop/direct/P57
✅ Head and Relation found in maps.

Predicted Tails (Directors?):
  1. [0.2180] http://www.wikidata.org/entity/Q3499135
  2. [0.2228] http://www.wikidata.org/entity/Q272076
  3. [0.2268] http://www.wikidata.org/entity/Q707796
  4. [0.2271] http://www.wikidata.org/entity/Q3241962
  5. [0.2391] http://www.wikidata.org/entity/Q1369224

💡 Logic Check: If you see entities above, the 'I think...' logic SHOULD work.
If the bot didn't output this, the bug is in 'nlq.py' (maybe predicate matching failed).


# Data Check for the example questions

In [3]:
import json
import os
import numpy as np
import faiss
from pathlib import Path
from rdflib import Graph, URIRef

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
CODE_DIR = Path("/files/Final Project Submission")

# Paths
GRAPH_PATH = DATA_DIR / "graph.nt"
IMAGES_PATH = DATA_DIR / "additional/images.json"
ENTITY_ID_PATH = DATA_DIR / "embeddings/entity_ids.del"
RELATION_ID_PATH = DATA_DIR / "embeddings/relation_ids.del"
ENT_EMB_PATH = CODE_DIR / "embeddings/RFC_entity_embeds.npy"
REL_EMB_PATH = CODE_DIR / "embeddings/RFC_relation_embeds.npy"

# ==========================================
# 2. LOAD ALL DATA
# ==========================================
print("--- Loading Data Resources ---")

# A. Load Graph
print(f"Loading Graph from {GRAPH_PATH}...")
g = Graph()
g.parse(GRAPH_PATH, format="nt")
print(f"✅ Graph loaded: {len(g)} triples")

# B. Load ID Mappings
print("Loading ID Mappings...")
ent_map = {}
idx2ent = {}
with open(ENTITY_ID_PATH, "r") as f:
    for line in f:
        i, u = line.strip().split('\t')
        ent_map[u] = int(i)
        idx2ent[int(i)] = u

rel_map = {}
with open(RELATION_ID_PATH, "r") as f:
    for line in f:
        i, u = line.strip().split('\t')
        rel_map[u] = int(i)

# C. Load Embeddings
print("Loading RFC Embeddings...")
if ENT_EMB_PATH.exists() and REL_EMB_PATH.exists():
    E = np.load(ENT_EMB_PATH)
    R = np.load(REL_EMB_PATH)
    
    # Build Search Index
    print("Building FAISS Index...")
    d = E.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(E)
    embeddings_ready = True
else:
    print("❌ Embeddings not found. Skipping embedding checks.")
    embeddings_ready = False

# D. Load Images
print("Loading Images...")
if IMAGES_PATH.exists():
    with open(IMAGES_PATH, "r") as f:
        img_data = json.load(f)
    print(f"✅ Loaded {len(img_data)} images")
else:
    img_data = []
    print("❌ Images file not found")

# ==========================================
# 3. TEST CASES (From Final Evaluation.txt)
# ==========================================
# Format: (Question Type, Entity Name, Property URI, Property Name)
test_cases = [
    # 1. Factual Questions (Graph should handle these)
    ("Fact", "Aro Tolbukhin. En la mente del asesino", "http://www.wikidata.org/prop/direct/P495", "Country"),
    ("Fact", "Shortcut to Happiness", "http://www.wikidata.org/prop/direct/P58", "Screenwriter"),
    ("Fact", "Fargo", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Fact", "Bandit Queen", "http://www.wikidata.org/prop/direct/P136", "Genre"),
    ("Fact", "Miracles Still Happen", "http://www.wikidata.org/prop/direct/P577", "Pub Date"),
    ("Fact", "Enchanted April", "http://www.wikidata.org/prop/direct/P86", "Composer"),
    ("Fact", "Midnight in Paris", "http://www.wikidata.org/prop/direct/P1411", "Nominated For"),
    ("Fact", "Garden State", "http://www.wikidata.org/prop/direct/P57", "Director"),

    # 2. Embedding "I Think" Questions (Graph might miss these, Embeddings must guess)
    ("Embedding", "Apocalypse Now", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Embedding", "12 Monkeys", "http://www.wikidata.org/prop/direct/P58", "Screenwriter"),
    ("Embedding", "Shoplifters", "http://www.wikidata.org/prop/direct/P136", "Genre"),
    ("Embedding", "Good Will Hunting", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Embedding", "Forrest Gump", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Embedding", "The Longest Day", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Embedding", "Captain America: Civil War", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Embedding", "The Ascent", "http://www.wikidata.org/prop/direct/P57", "Director"),
]

multimedia_cases = [
    "Halle Berry", "Denzel Washington", "Forrest Gump"
]

# ==========================================
# 4. EXECUTE DIAGNOSTICS
# ==========================================
print("\n" + "="*60)
print("STARTING DEEP DIVE DIAGNOSIS")
print("="*60)

for q_type, name, pred_uri, pred_name in test_cases:
    print(f"\n🔹 [{q_type}] Checking: {name} -> {pred_name}...")
    
    # --- A. Find Entity URI ---
    query_label = f"""SELECT ?s WHERE {{ ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label . FILTER(LCASE(STR(?label)) = LCASE("{name}")) }}"""
    res = list(g.query(query_label))
    
    if not res:
        print(f"   ❌ Entity NOT FOUND in Graph")
        continue
    
    entity_uri = str(res[0][0])
    print(f"   ✅ URI: {entity_uri}")

    # --- B. Check Graph Answer (Ground Truth) ---
    query_fact = f"""SELECT ?oLabel WHERE {{ <{entity_uri}> <{pred_uri}> ?o . ?o <http://www.w3.org/2000/01/rdf-schema#label> ?oLabel . FILTER(LANG(?oLabel) = 'en') }}"""
    fact_res = list(g.query(query_fact))
    
    graph_has_answer = False
    if fact_res:
        answers = [str(r[0]) for r in fact_res]
        print(f"   ✅ Graph Answer: {', '.join(answers)}")
        graph_has_answer = True
    else:
        print(f"   ⚠️  Graph Answer MISSING")

    # --- C. Check Embedding Prediction (The 'I Think' Logic) ---
    if embeddings_ready:
        if entity_uri in ent_map and pred_uri in rel_map:
            h_idx = ent_map[entity_uri]
            r_idx = rel_map[pred_uri]
            
            # Predict: Head + Relation
            vec = E[h_idx] + R[r_idx]
            D, I = index.search(vec.reshape(1, -1), 5)
            
            top_preds = []
            for idx in I[0]:
                uri = idx2ent.get(idx, "")
                # Resolve Label for better readability
                lbl_res = list(g.query(f"""SELECT ?l WHERE {{ <{uri}> <http://www.w3.org/2000/01/rdf-schema#label> ?l . FILTER(LANG(?l)='en')}}"""))
                lbl = str(lbl_res[0][0]) if lbl_res else uri.split('/')[-1]
                top_preds.append(lbl)
            
            print(f"   🧠 Embedding Prediction (Top 3): {', '.join(top_preds[:3])}")
            
            if not graph_has_answer:
                print("      -> Embedding fills the gap! This confirms 'I think' logic will work.")
        else:
            print("   ❌ Entity/Relation not in Embedding Map")

print("\n" + "="*60)
print("MULTIMEDIA CHECK")
print("="*60)

for name in multimedia_cases:
    found = False
    print(f"\n🖼️ Checking Image for '{name}'...")
    
    # Find URI first
    query_label = f"""SELECT ?s WHERE {{ ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label . FILTER(LCASE(STR(?label)) = LCASE("{name}")) }}"""
    res = list(g.query(query_label))
    if not res:
        print("   ❌ Entity URI not found")
        continue
    uri = str(res[0][0]).strip("<>")
    qid = uri.split("/")[-1]
    
    # Check JSON
    for item in img_data:
        uris = []
        for k in ['movie', 'cast', 'id']:
            val = item.get(k)
            if isinstance(val, list): uris.extend(val)
            elif isinstance(val, str): uris.append(val)
        
        # Check both full URI and QID
        if uri in uris or qid in uris:
            print(f"   ✅ Image Found: {item.get('img')}")
            found = True
            break
            
    if not found:
        # Check for IMDb bridge
        query_imdb = f"""SELECT ?id WHERE {{ <{uri}> <http://www.wikidata.org/prop/direct/P345> ?id }}"""
        imdb_res = list(g.query(query_imdb))
        if imdb_res:
            imdb_id = str(imdb_res[0][0])
            print(f"   ℹ️  Found IMDb ID: {imdb_id}. Checking JSON...")
            for item in img_data:
                uris = []
                for k in ['movie', 'cast', 'id']:
                    val = item.get(k)
                    if isinstance(val, list): uris.extend(val)
                    elif isinstance(val, str): uris.append(val)
                
                if imdb_id in uris:
                    print(f"   ✅ Image Found via IMDb: {item.get('img')}")
                    found = True
                    break
        
    if not found:
        print("   ❌ No image found (Direct or via IMDb). Data is missing.")

--- Loading Data Resources ---
Loading Graph from /space_mounts/atai-hs25/dataset/graph.nt...
✅ Graph loaded: 2413825 triples
Loading ID Mappings...
Loading RFC Embeddings...
Building FAISS Index...
Loading Images...
✅ Loaded 347496 images

STARTING DEEP DIVE DIAGNOSIS

🔹 [Fact] Checking: Aro Tolbukhin. En la mente del asesino -> Country...
   ✅ URI: http://www.wikidata.org/entity/Q4795458
   ⚠️  Graph Answer MISSING
   🧠 Embedding Prediction (Top 3): Q298, Q29, Q77
      -> Embedding fills the gap! This confirms 'I think' logic will work.

🔹 [Fact] Checking: Shortcut to Happiness -> Screenwriter...
   ✅ URI: http://www.wikidata.org/entity/Q1115508
   ⚠️  Graph Answer MISSING
   🧠 Embedding Prediction (Top 3): Q1750712, Q361336, Q287607
      -> Embedding fills the gap! This confirms 'I think' logic will work.

🔹 [Fact] Checking: Fargo -> Director...
   ✅ URI: http://www.wikidata.org/entity/Q222720
   ⚠️  Graph Answer MISSING
   🧠 Embedding Prediction (Top 3): Q350422, Q2563751, Q42574

# Data Extraction

In [4]:
import json
import os
from pathlib import Path

# --- CONFIGURATION ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
METADATA_DIR = Path("metadata")
METADATA_DIR.mkdir(exist_ok=True)
GRAPH_PATH = DATA_DIR / "graph.nt"

# Standard URIs
P_LABEL = "http://www.w3.org/2000/01/rdf-schema#label"
P_IMDB = "http://www.wikidata.org/prop/direct/P345"

print(f"Scanning {GRAPH_PATH} with STRICT CLEANING...\n")

entity_labels = {}
imdb_map = {}

# Counters
count_labels = 0
count_imdb = 0

with open(GRAPH_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(" ")
        if len(parts) < 3: continue
        
        s = parts[0].strip("<>")
        p = parts[1].strip("<>")
        
        # --- 1. Label Extraction ---
        if p == P_LABEL:
            # Reconstruct object string (it might contain spaces)
            # parts[2:] contains "Name"@en .
            obj_raw = " ".join(parts[2:-1]) 
            
            if '"' in obj_raw:
                # Extract content between first and last quote to handle names with quotes
                start = obj_raw.find('"') + 1
                end = obj_raw.rfind('"')
                if end > start:
                    label = obj_raw[start:end]
                    
                    # Logic: Prefer @en, but take what we can get
                    if "@en" in obj_raw:
                        entity_labels[s] = label
                    elif s not in entity_labels:
                        entity_labels[s] = label
                        
        # --- 2. IMDb ID Extraction (CLEANING FIX) ---
        if p == P_IMDB:
            o_raw = parts[2]
            # o_raw looks like: "tt12345"^^<http://...>
            
            if '"' in o_raw:
                # Extract exactly what is inside the quotes
                # "tt12345" -> tt12345
                imdb_id = o_raw.split('"')[1]
                imdb_map[s] = imdb_id
                count_imdb += 1

print(f"✅ Extracted {len(entity_labels)} labels.")
print(f"✅ Extracted {len(imdb_map)} CLEAN IMDb mappings.")

# --- Save to JSON ---
with open(METADATA_DIR / "entity_labels.json", "w") as f:
    json.dump(entity_labels, f)
with open(METADATA_DIR / "imdb_map.json", "w") as f:
    json.dump(imdb_map, f)
    
print(f"Saved cleaned metadata to {METADATA_DIR}")

# --- Verification ---
test_uris = ["http://www.wikidata.org/entity/Q1033016", "http://www.wikidata.org/entity/Q134773"]
print("\nVerifying Clean IDs:")
for uri in test_uris:
    if uri in imdb_map:
        print(f"  {uri} -> {imdb_map[uri]}")
    else:
        print(f"  {uri} -> MISSING")

Scanning /space_mounts/atai-hs25/dataset/graph.nt with STRICT CLEANING...

✅ Extracted 129673 labels.
✅ Extracted 95569 CLEAN IMDb mappings.
Saved cleaned metadata to metadata

Verifying Clean IDs:
  http://www.wikidata.org/entity/Q1033016 -> nm0000932
  http://www.wikidata.org/entity/Q134773 -> tt0109830


# Final End-to-End Validation

In [7]:
import json
import os
import numpy as np
import faiss
import re
from pathlib import Path
from rdflib import Graph, URIRef
from rapidfuzz import process, fuzz

# ==========================================
# 1. SETUP & MOCK CLASSES (Simulating your Agent)
# ==========================================
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
CODE_DIR = Path("/files/Final Project Submission")
METADATA_DIR = Path("metadata")

# Load Data
print("--- Loading Resources ---")
GRAPH_PATH = DATA_DIR / "graph.nt"
g = Graph()
g.parse(GRAPH_PATH, format="nt")
print(f"✅ Graph: {len(g)} triples")

with open(METADATA_DIR / "entity_labels.json", "r") as f:
    labels = json.load(f)
with open(METADATA_DIR / "imdb_map.json", "r") as f:
    imdb_map = json.load(f)
with open(DATA_DIR / "additional/images.json", "r") as f:
    img_db = json.load(f)

# Load Embeddings
E = np.load(CODE_DIR / "embeddings/RFC_entity_embeds.npy")
R = np.load(CODE_DIR / "embeddings/RFC_relation_embeds.npy")

# Mappings
ent_map = {}
idx2ent = {}
with open(DATA_DIR / "embeddings/entity_ids.del") as f:
    for line in f:
        i, u = line.strip().split('\t')
        ent_map[u] = int(i)
        idx2ent[int(i)] = u

rel_map = {}
with open(DATA_DIR / "embeddings/relation_ids.del") as f:
    for line in f:
        i, u = line.strip().split('\t')
        rel_map[u] = int(i)

# FAISS Index
index = faiss.IndexFlatL2(E.shape[1])
index.add(E)

# --- MOCK COMPONENTS ---

def mock_link(text):
    # Simplified version of your EntityLinker logic
    # 1. Try exact match in labels
    text_lower = text.lower()
    best_match = None
    best_score = 0
    
    # Fast scan (in real bot this is optimized)
    for uri, lbl in labels.items():
        if lbl.lower() == text_lower:
            return uri, lbl
        
        # Fuzzy
        score = fuzz.ratio(lbl.lower(), text_lower)
        if score > best_score:
            best_score = score
            best_match = (uri, lbl)
            
    if best_score > 85:
        return best_match
    return None, None

def mock_get_image(uri):
    # Logic from your MultimediaIndex
    clean_uri = uri.strip("<>")
    
    # 1. Check direct QID
    qid = clean_uri.split("/")[-1]
    
    # 2. Check IMDb
    imdb_id = imdb_map.get(clean_uri)
    
    for item in img_db:
        vals = []
        for k in ['movie', 'cast', 'id']:
            v = item.get(k)
            if isinstance(v, list): vals.extend(v)
            elif isinstance(v, str): vals.append(v)
            
        if clean_uri in vals or qid in vals or (imdb_id and imdb_id in vals):
            return f"image:{item['img'][:-4]}" if item['img'].endswith('.jpg') else f"image:{item['img']}"
    return None

def mock_qa(entity_name, pred_uri, pred_name):
    print(f"\n❓ Q: What is {pred_name} of '{entity_name}'?")
    
    # 1. Link
    uri, label = mock_link(entity_name)
    if not uri:
        print("   ❌ Entity Link Failed")
        return

    print(f"   ✅ Linked: {label} ({uri})")
    
    # 2. Try Graph (One-hop)
    query = f"""SELECT ?oLabel WHERE {{ <{uri}> <{pred_uri}> ?o . ?o <http://www.w3.org/2000/01/rdf-schema#label> ?oLabel . FILTER(LANG(?oLabel)='en') }}"""
    res = list(g.query(query))
    
    if res:
        ans = [str(r[0]) for r in res]
        print(f"   ✅ GRAPH ANSWER: {', '.join(ans)}")
        return

    # 3. Try Embedding (I think...)
    print("   ⚠️  Graph Empty. Trying Embedding...")
    if uri in ent_map and pred_uri in rel_map:
        h = ent_map[uri]
        r = rel_map[pred_uri]
        pred_vec = E[h] + R[r]
        D, I = index.search(pred_vec.reshape(1, -1), 3)
        
        preds = []
        for idx in I[0]:
            u = idx2ent[idx]
            l = labels.get(u, u.split('/')[-1])
            preds.append(l)
        print(f"   🧠 EMBEDDING GUESS: I think {', '.join(preds)}")
    else:
        print("   ❌ Embedding Missing (Entity or Relation not in map)")

def mock_rec(seed_names):
    print(f"\n🎬 Recommend similar to: {seed_names}")
    seeds = []
    for name in seed_names:
        u, l = mock_link(name)
        if u: seeds.append(u)
    
    if not seeds:
        print("   ❌ No seeds identified.")
        return

    # Mock embedding nearest neighbors
    print("   ...Calculating neighbors...")
    candidates = {}
    for seed in seeds:
        if seed not in ent_map: continue
        idx = ent_map[seed]
        D, I = index.search(E[idx].reshape(1, -1), 10)
        for i in I[0]:
            u = idx2ent[i]
            if u not in seeds:
                candidates[u] = candidates.get(u, 0) + 1
    
    # Sort and print top 5
    sorted_cands = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:5]
    for u, score in sorted_cands:
        l = labels.get(u, "Unknown")
        print(f"   -> {l}")

# ==========================================
# 2. RUNNING THE EVALUATION CHECKLIST
# ==========================================

print("\n" + "="*50)
print("SECTION 1: ONE-HOP QA CHECK")
print("="*50)

# Property URIs
P_DIR = "http://www.wikidata.org/prop/direct/P57"
P_SCREEN = "http://www.wikidata.org/prop/direct/P58"
P_COUNTRY = "http://www.wikidata.org/prop/direct/P495"
P_GENRE = "http://www.wikidata.org/prop/direct/P136"
P_DATE = "http://www.wikidata.org/prop/direct/P577"
P_COMP = "http://www.wikidata.org/prop/direct/P86"
P_AWARD = "http://www.wikidata.org/prop/direct/P166" # Nominated is P1411 but examples use won/nom mixed

qa_list = [
    ("Aro Tolbukhin. En la mente del asesino", P_COUNTRY, "Country"),
    ("Shortcut to Happiness", P_SCREEN, "Screenwriter"),
    ("Fargo", P_DIR, "Director"),
    ("Bandit Queen", P_GENRE, "Genre"),
    ("Miracles Still Happen", P_DATE, "Publication Date"),
    ("Apocalypse Now", P_DIR, "Director"),
    ("12 Monkeys", P_SCREEN, "Screenwriter"),
    ("Shoplifters", P_GENRE, "Genre"),
    ("Good Will Hunting", P_DIR, "Director"),
    ("Garden State", P_DIR, "Director"),
    ("Enchanted April", P_COMP, "Composer"),
    ("Midnight in Paris", "http://www.wikidata.org/prop/direct/P1411", "Nominated For"),
    ("Forrest Gump", P_DIR, "Director"),
    ("The Longest Day", P_DIR, "Director"),
    ("Captain America: Civil War", P_DIR, "Director"),
    ("The Ascent", P_DIR, "Director")
]

for ent, prop, pname in qa_list:
    mock_qa(ent, prop, pname)


print("\n" + "="*50)
print("SECTION 2: RECOMMENDATION CHECK")
print("="*50)

rec_list = [
    ["Singin' in the Rain", "Moulin Rouge"],
    ["La Dolce Vita", "The Voice of the Moon"],
    ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    ["Twin Sisters of Kyoto"],
    ["Meryl Streep"]
]

for seeds in rec_list:
    mock_rec(seeds)


print("\n" + "="*50)
print("SECTION 3: MULTIMEDIA CHECK")
print("="*50)

mm_list = ["Halle Berry", "Denzel Washington", "Forest Gump"]

for name in mm_list:
    print(f"\n🖼️ Image for '{name}'?")
    # Try fuzzy link first because "Forest" is a typo of "Forrest"
    uri, lbl = mock_link(name)
    
    if not uri and name == "Forest Gump":
        # Simulate typo fix manually if fuzzy fails
        uri, lbl = "http://www.wikidata.org/entity/Q134773", "Forrest Gump"
    
    if uri:
        img = mock_get_image(uri)
        if img:
            print(f"   ✅ Found: {img}")
        else:
            print(f"   ❌ Image missing in DB for {lbl}")
    else:
        print("   ❌ Entity not found")

--- Loading Resources ---
✅ Graph: 2413825 triples

SECTION 1: ONE-HOP QA CHECK

❓ Q: What is Country of 'Aro Tolbukhin. En la mente del asesino'?
   ✅ Linked: Aro Tolbukhin. En la mente del asesino (http://www.wikidata.org/entity/Q4795458)
   ⚠️  Graph Empty. Trying Embedding...
   🧠 EMBEDDING GUESS: I think Chile, Spain, Uruguay

❓ Q: What is Screenwriter of 'Shortcut to Happiness'?
   ✅ Linked: Shortcut to Happiness (http://www.wikidata.org/entity/Q1115508)
   ⚠️  Graph Empty. Trying Embedding...
   🧠 EMBEDDING GUESS: I think Scott Cooper, Q361336, Rob Reiner

❓ Q: What is Director of 'Fargo'?
   ✅ Linked: Fargo (http://www.wikidata.org/entity/Q222720)
   ⚠️  Graph Empty. Trying Embedding...
   🧠 EMBEDDING GUESS: I think Wilter Hall, Jon Peters, James Cameron

❓ Q: What is Genre of 'Bandit Queen'?
   ✅ Linked: Bandit Queen (http://www.wikidata.org/entity/Q2762808)
   ⚠️  Graph Empty. Trying Embedding...
   🧠 EMBEDDING GUESS: I think Masala film, political film, psychological thrille

In [8]:
import json
import numpy as np
import faiss
from pathlib import Path
from rdflib import Graph

# --- SETUP ---
DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
CODE_DIR = Path("/files/Final Project Submission")
METADATA_DIR = Path("metadata")

GRAPH_PATH = DATA_DIR / "graph.nt"
ENTITY_LABELS_PATH = METADATA_DIR / "entity_labels.json"

print("Loading Graph...")
g = Graph()
g.parse(GRAPH_PATH, format="nt")

print("Loading Labels Dictionary...")
with open(ENTITY_LABELS_PATH, "r") as f:
    # Load all labels into memory for fast lookup
    # Format: {"http://.../Q123": "Label"}
    id_to_label = json.load(f)

print("Loading Embeddings...")
ENT_EMB_PATH = CODE_DIR / "embeddings/RFC_entity_embeds.npy"
REL_EMB_PATH = CODE_DIR / "embeddings/RFC_relation_embeds.npy"
E = np.load(ENT_EMB_PATH)
R = np.load(REL_EMB_PATH)

# Load ID Maps
ent_map = {}
idx2ent = {}
with open(DATA_DIR / "embeddings/entity_ids.del") as f:
    for line in f:
        i, u = line.strip().split('\t')
        ent_map[u] = int(i)
        idx2ent[int(i)] = u

rel_map = {}
with open(DATA_DIR / "embeddings/relation_ids.del") as f:
    for line in f:
        i, u = line.strip().split('\t')
        rel_map[u] = int(i)

# Build FAISS
index = faiss.IndexFlatL2(E.shape[1])
index.add(E)

# --- HELPER FUNCTIONS ---

def get_label(uri):
    """Safe label lookup from our pre-computed dictionary"""
    # Try full URI
    if uri in id_to_label: return id_to_label[uri]
    # Try QID
    qid = uri.split("/")[-1]
    if qid in id_to_label: return id_to_label[qid]
    return qid # Fallback to ID

def solve_qa(entity_name, prop_uri, prop_name):
    print(f"\n❓ Q: What is {prop_name} of '{entity_name}'?")
    
    # 1. LINKING (Simulated)
    # Find URI matching the name in our label dict
    entity_uri = None
    for uri, label in id_to_label.items():
        if label.lower() == entity_name.lower():
            entity_uri = uri
            break
    
    if not entity_uri:
        print("   ❌ Linker: Entity not found in labels.")
        return

    print(f"   ✅ Linked: {entity_uri}")

    # 2. FACTUAL LOOKUP (Graph)
    # Simple query: Get the Object ID only (don't join label yet)
    query = f"SELECT ?o WHERE {{ <{entity_uri}> <{prop_uri}> ?o . }}"
    res = list(g.query(query))
    
    if res:
        # We found IDs! Now lookup their names
        answers = []
        for r in res:
            obj_uri = str(r[0])
            name = get_label(obj_uri)
            answers.append(name)
        
        print(f"   ✅ FACTUAL ANSWER: {', '.join(answers)}")
        return

    # 3. EMBEDDING INFERENCE (Fallback)
    # Only runs if Graph returned nothing
    print("   ⚠️  Fact Missing. Using Embedding...")
    if entity_uri in ent_map and prop_uri in rel_map:
        h_idx = ent_map[entity_uri]
        r_idx = rel_map[prop_uri]
        pred_vec = E[h_idx] + R[r_idx]
        D, I = index.search(pred_vec.reshape(1, -1), 3)
        
        preds = [get_label(idx2ent[i]) for i in I[0]]
        print(f"   🧠 PREDICTION: I think {', '.join(preds)}")
    else:
        print("   ❌ Embeddings cannot run (IDs missing from map).")

# --- EXECUTE TEST CASES ---

qa_cases = [
    # Should be FACTUAL
    ("Aro Tolbukhin. En la mente del asesino", "http://www.wikidata.org/prop/direct/P495", "Country"),
    ("Shortcut to Happiness", "http://www.wikidata.org/prop/direct/P58", "Screenwriter"),
    ("Fargo", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Bandit Queen", "http://www.wikidata.org/prop/direct/P136", "Genre"),
    ("Miracles Still Happen", "http://www.wikidata.org/prop/direct/P577", "Pub Date"),
    ("Good Will Hunting", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Garden State", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Enchanted April", "http://www.wikidata.org/prop/direct/P86", "Composer"),
    ("Midnight in Paris", "http://www.wikidata.org/prop/direct/P1411", "Nominated For"),
    
    # Should be EMBEDDING ("I think...")
    ("Apocalypse Now", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("12 Monkeys", "http://www.wikidata.org/prop/direct/P58", "Screenwriter"),
    ("Shoplifters", "http://www.wikidata.org/prop/direct/P136", "Genre"),
    ("Forrest Gump", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("The Longest Day", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("Captain America: Civil War", "http://www.wikidata.org/prop/direct/P57", "Director"),
    ("The Ascent", "http://www.wikidata.org/prop/direct/P57", "Director"),
]

for name, prop, pname in qa_cases:
    solve_qa(name, prop, pname)

Loading Graph...
Loading Labels Dictionary...
Loading Embeddings...

❓ Q: What is Country of 'Aro Tolbukhin. En la mente del asesino'?
   ✅ Linked: http://www.wikidata.org/entity/Q4795458
   ✅ FACTUAL ANSWER: Mexico

❓ Q: What is Screenwriter of 'Shortcut to Happiness'?
   ✅ Linked: http://www.wikidata.org/entity/Q1115508
   ✅ FACTUAL ANSWER: Pete Dexter, Q361336

❓ Q: What is Director of 'Fargo'?
   ✅ Linked: http://www.wikidata.org/entity/Q222720
   ✅ FACTUAL ANSWER: Ethan Coen, Joel Coen

❓ Q: What is Genre of 'Bandit Queen'?
   ✅ Linked: http://www.wikidata.org/entity/Q2762808
   ✅ FACTUAL ANSWER: drama film, biographical film, crime film

❓ Q: What is Pub Date of 'Miracles Still Happen'?
   ✅ Linked: http://www.wikidata.org/entity/Q5907652
   ✅ FACTUAL ANSWER: 1974-07-19

❓ Q: What is Director of 'Good Will Hunting'?
   ✅ Linked: http://www.wikidata.org/entity/Q193835
   ✅ FACTUAL ANSWER: Gus Van Sant

❓ Q: What is Director of 'Garden State'?
   ✅ Linked: http://www.wikidata.org/e